<a href="https://colab.research.google.com/github/NeuroNex-ModelX/drainage-failure-sri-lanka/blob/main/02_rainfall_chirps.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · Rainfall (CHIRPS v2.0 daily) → rainfall features per DS division × month
**Owner:** Prameshka · **Needs:** notebook 00 · **Makes:** `rain_daily.csv.gz`, `rain_monthly_features.csv`

CHIRPS is read from **Google Earth Engine** (official copy: `UCSB-CHG/CHIRPS/DAILY`). The CHC download
server blocks automated downloads (HTTP 403, robots.txt), so we do not scrape it. Earth Engine averages
the ~5 km rainfall over each DS division on Google's servers, so nothing large is downloaded.

**One-time setup (5 min, only the person running this notebook):**
1. Open https://code.earthengine.google.com/register (use the same Google account as Colab).
2. Choose **Use with a noncommercial / unpaid project** → **Academia & research** (student).
3. **Create a new Google Cloud project**, e.g. `neuronex-modelx` → Continue → Confirm.
4. Put that project ID in `EE_PROJECT` below.

**Runtime:** about 1 min per year (~30 min for 1995–2025). Re-running skips years already saved.

In [ ]:
# --- SETUP ---
import os, sys, json, time
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    os.environ['MODELX_BASE'] = '/content/drive/MyDrive/ModelX'
sys.path.append(os.path.join(os.environ.get('MODELX_BASE', '..'), 'src'))
from common import *

EE_PROJECT = 'neuronex-modelx'      # <-- your Google Cloud project ID from the setup above
START_YEAR, END_YEAR = 1995, 2025   # proposal: history from 1995; 2024-2025 used for testing
year_dir = os.path.join(PROC, 'chirps_ds_daily')   # one small CSV per year
os.makedirs(year_dir, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import ee
ee.Authenticate()                   # follow the pop-up / link the first time
ee.Initialize(project=EE_PROJECT)
print('Earth Engine ready')

Earth Engine ready


### Step 1 – send the DS-division shapes to Earth Engine

In [ ]:
import geopandas as gpd

ds_poly = gpd.read_file(os.path.join(PROC, 'ds_divisions.gpkg'))[['pcode', 'geometry']]
simple = ds_poly.copy()
# simplify (~100 m) so the request is small; buffer(0) repairs any invalid shapes
simple['geometry'] = simple.to_crs(SL_CRS).simplify(100).buffer(0).to_crs('EPSG:4326')
fc = ee.FeatureCollection(json.loads(simple.to_json()))
print(fc.size().getInfo(), 'DS divisions sent to Earth Engine')

339 DS divisions sent to Earth Engine


### Step 2 – daily mean rainfall per DS division, one year at a time
`reduceRegions` averages all CHIRPS cells overlapping each division (weighted by overlap),
so even small divisions get a value.

In [ ]:
CHIRPS = ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY').select('precipitation')


def fetch(start, end):
    """Daily mean rain (mm) per division between start (incl.) and end (excl.) -> long DataFrame."""
    img = CHIRPS.filterDate(start, end).toBands()            # one band per day: 'YYYYMMDD_precipitation'
    res = img.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=5566).getInfo()
    rows = []
    for f in res['features']:
        p = f['properties']
        for k, v in p.items():
            m = re.fullmatch(r'(\d{8})_precipitation', k)
            if m:
                rows.append((p['pcode'], m.group(1), v))
    out = pd.DataFrame(rows, columns=['pcode', 'date', 'rain_mm'])
    out['date'] = pd.to_datetime(out['date'], format='%Y%m%d')
    return out


for y in range(START_YEAR, END_YEAR + 1):
    path = os.path.join(year_dir, f'rain_{y}.csv')
    if os.path.exists(path):
        print(y, 'already done'); continue
    t0 = time.time()
    try:
        df = fetch(f'{y}-01-01', f'{y + 1}-01-01')
    except Exception as e:           # if a whole year is too big, do it month by month
        print(f'   whole year failed ({str(e)[:80]}) - trying month by month')
        parts = []
        for mth in range(1, 13):
            a = f'{y}-{mth:02d}-01'
            b = f'{y + 1}-01-01' if mth == 12 else f'{y}-{mth + 1:02d}-01'
            parts.append(fetch(a, b))
        df = pd.concat(parts)
    if df.empty:
        print(y, 'no data returned (year not published yet?)'); continue
    df.to_csv(path, index=False)
    print(f"{y}: {df['date'].nunique()} days x {df['pcode'].nunique()} divisions ({time.time() - t0:.0f}s)")

save_meta('rainfall', f'CHIRPS v2.0 daily 0.05 deg via Google Earth Engine (UCSB-CHG/CHIRPS/DAILY), {START_YEAR}-{END_YEAR}')

1995: 365 days x 339 divisions (9s)
1996: 366 days x 339 divisions (9s)
1997: 365 days x 339 divisions (11s)
1998: 365 days x 339 divisions (11s)
1999: 365 days x 339 divisions (9s)
2000: 366 days x 339 divisions (10s)
2001: 365 days x 339 divisions (9s)
2002: 365 days x 339 divisions (13s)
2003: 365 days x 339 divisions (12s)
2004: 366 days x 339 divisions (13s)
2005: 365 days x 339 divisions (13s)
2006: 365 days x 339 divisions (10s)
2007: 365 days x 339 divisions (12s)
2008: 366 days x 339 divisions (10s)
2009: 365 days x 339 divisions (10s)
2010: 365 days x 339 divisions (11s)
2011: 365 days x 339 divisions (11s)
2012: 366 days x 339 divisions (11s)
2013: 365 days x 339 divisions (13s)
2014: 365 days x 339 divisions (10s)
2015: 365 days x 339 divisions (10s)
2016: 366 days x 339 divisions (13s)
2017: 365 days x 339 divisions (9s)
2018: 365 days x 339 divisions (11s)
2019: 365 days x 339 divisions (14s)
2020: 366 days x 339 divisions (12s)
2021: 365 days x 339 divisions (16s)
2022: 

{'boundaries': 'Survey Dept of Sri Lanka via OCHA/HDX (cod-ab-lka), CC BY-IGO',
 '_updated': '2026-10-29 15:13',
 'flood_records': 'DesInventar Sri Lanka, 1974-2013',
 'flood_last_year': 2013,
 'rainfall': 'CHIRPS v2.0 daily 0.05 deg via Google Earth Engine (UCSB-CHG/CHIRPS/DAILY), 1995-2025'}

### Step 3 – combine the years into one daily table

In [ ]:
files = sorted(find_files(year_dir, [r'rain_\d{4}\.csv$']))
rain_daily = pd.concat([pd.read_csv(f, parse_dates=['date']) for f in files], ignore_index=True)
rain_daily = rain_daily.sort_values(['pcode', 'date'])
missing = rain_daily['rain_mm'].isna().mean()
print(f'{len(files)} years | {rain_daily.shape[0]:,} rows | missing values: {missing:.2%}')
rain_daily.to_csv(os.path.join(PROC, 'rain_daily.csv.gz'), index=False)
print(rain_daily['date'].min().date(), '->', rain_daily['date'].max().date())

31 years | 3,838,497 rows | missing values: 0.00%
1995-01-01 -> 2025-12-31


### Step 4 – monthly features (one row per division × month)
monthly total, wettest day, days > 50 mm, days > 100 mm, previous month's total.

In [ ]:
r = rain_daily.copy()
r['year'], r['month'] = r['date'].dt.year, r['date'].dt.month
r['over_50'] = (r['rain_mm'] > 50).astype(int)
r['over_100'] = (r['rain_mm'] > 100).astype(int)
feat = (r.groupby(['pcode', 'year', 'month'])
        .agg(rain_total=('rain_mm', 'sum'), rain_max_day=('rain_mm', 'max'),
             days_over_50=('over_50', 'sum'), days_over_100=('over_100', 'sum'))
        .reset_index().sort_values(['pcode', 'year', 'month']))
feat['rain_prev_month'] = feat.groupby('pcode')['rain_total'].shift(1)
feat.to_csv(os.path.join(PROC, 'rain_monthly_features.csv'), index=False)
print(feat.shape)
feat.describe().round(1)

(126108, 8)


,year,month,rain_total,rain_max_day,days_over_50,days_over_100,rain_prev_month
count,126108.0,126108.0,126108.0,126108.0,126108.0,126108.0,125769.0
mean,2010.0,6.5,180.8,33.2,0.4,0.0,180.9
std,8.9,3.5,155.7,24.4,0.9,0.2,155.9
min,1995.0,1.0,0.0,0.0,0.0,0.0,0.0
25%,2002.0,3.8,60.2,15.9,0.0,0.0,60.0
50%,2010.0,6.5,141.3,28.3,0.0,0.0,141.1
75%,2018.0,9.2,262.4,44.5,0.0,0.0,262.8
max,2025.0,12.0,1383.0,251.8,13.0,5.0,1383.0


In [ ]:
# Sanity check (good figure for the report): wettest divisions around Cyclone Ditwah
ditwah = rain_daily[(rain_daily['date'] >= '2025-11-15') & (rain_daily['date'] <= '2025-12-10')]
top = ditwah.groupby('pcode')['rain_mm'].sum().sort_values(ascending=False).head(10)
names = load_ds_list().set_index('pcode')['ds_name']
print('Wettest divisions, 15 Nov - 10 Dec 2025 (mm):')
print(top.rename(index=names).round(0))

Wettest divisions, 15 Nov - 10 Dec 2025 (mm):
pcode
Kotapola             706.0
Pasgoda              671.0
Puthukkudiyiruppu    665.0
Maritimepattu        659.0
Neluwa               635.0
Minipe               635.0
Pitabaddara          634.0
Oddusuddan           625.0
Kalawana             622.0
Thawalama            619.0
Name: rain_mm, dtype: float64


### Optional – cross-check with GPM IMERG for Cyclone Ditwah (also from Earth Engine)
Compares IMERG and CHIRPS event totals per division (proposal: IMERG to cross-check major events).

In [ ]:
try:
    imerg = (ee.ImageCollection('NASA/GPM_L3/IMERG_V07').filterDate('2025-11-15', '2025-12-16')
             .select('precipitation').sum().multiply(0.5))          # mm/hr every 30 min -> mm
    res = imerg.reduceRegions(collection=fc, reducer=ee.Reducer.mean(), scale=11132).getInfo()
    comp = pd.DataFrame([(f['properties']['pcode'], f['properties'].get('mean')) for f in res['features']],
                        columns=['pcode', 'imerg_mm'])
    span = (rain_daily['date'] >= '2025-11-15') & (rain_daily['date'] <= '2025-12-15')
    comp = comp.merge(rain_daily[span].groupby('pcode')['rain_mm'].sum().rename('chirps_mm'), on='pcode')
    print('Correlation CHIRPS vs IMERG (Ditwah event totals per division):',
          round(comp[['imerg_mm', 'chirps_mm']].corr().iloc[0, 1], 2))
    comp.to_csv(os.path.join(OUT, 'imerg_vs_chirps_ditwah.csv'), index=False)
except Exception as e:
    print('IMERG cross-check skipped:', str(e)[:150])

Correlation CHIRPS vs IMERG (Ditwah event totals per division): 0.37
